# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

In [ ]:
# List all record set `@id`s and details
record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    print(f"  Name: {rs.get('name', None)}")
    print(f"  Description: {rs.get('description', None)}")
    # List fields
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print(f"  Fields:")
        for f in fields:
            print(f"    Field @id: {f['@id']}, Name: {f.get('name', '')}")
    # List columns if present
    if 'column' in rs:
        columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
        print(f"  Columns:")
        for c in columns:
            print(f"    Column @id: {c['@id']}, Name: {c.get('name', '')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# This dataset contains one main record set: 'https://api.app.sen.science/frontiers/7160411/d17d577d-4aa6-4875-982a-30cae30b7bc4'
main_record_set_id = 'https://api.app.sen.science/frontiers/7160411/d17d577d-4aa6-4875-982a-30cae30b7bc4'

# For this demonstration, extract all available record sets.
record_sets_ids = [main_record_set_id]
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

df = dataframes[main_record_set_id]
print(f"Fields for {main_record_set_id}:")
print(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis; we'll use the GAD-7 total score, whose field @id is typically 'gad7_total_score'
numeric_field_id = 'gad7_total_score'
record_set_id = main_record_set_id

# Proceed if the field exists
if numeric_field_id in dataframes[record_set_id].columns:
    threshold = 10
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field_id].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a demographic field, e.g. 'gender'
    group_field_id = 'gender'  # Use @id for gender field (adjust if different)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df)
else:
    print(f"Field {numeric_field_id} not found in {record_set_id}.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    # Distribution of the GAD-7 score (gad7_total_score)
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].astype(float), bins=8, kde=True)
    plt.title('GAD-7 Total Score Distribution')
    plt.xlabel('GAD-7 Total Score')
    plt.ylabel('Number of Participants')
    plt.show()

    # Boxplot of GAD-7 score by gender
    if 'gender' in df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x='gender', y=numeric_field_id, data=df)
        plt.title('GAD-7 Total Score by Gender')
        plt.xlabel('Gender')
        plt.ylabel('GAD-7 Total Score')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides comprehensive survey data on mental health indicators in Kilifi County, Kenya, including scores from GAD-7, PHQ-9, and PSQ assessments, as well as rich demographic information.
- We demonstrated how to load data and metadata using the `mlcroissant` library and explored available record sets using their `@id`.
- Data fields such as GAD-7 total score were analyzed, with example filters and normalization procedures performed, and subgroup means demonstrated using the field `@id` conventions.
- Visualizations showed the score distributions and demographic breakdowns, helping reveal trends for further research and analysis.

For more advanced analysis, consider extending this notebook to explore PHQ-9, PSQ, or other demographic/symptom relationships.